In [ ]:
import pandas as pd
import nibabel as nib
import numpy as np
import yaml
import os
import sys
import argparse
import glob

In [3]:
ico_res = 6
ico_down_res = 2
down_num_vertices = 153 
down_num_patches = 320

## destination privacy etc? ask janine

subject_path = "/ceph/chpc/shared/janine_bijsterbosch_group/lucas_scratch/HCP_YA/resampled_icos"
patch_extraction_path = "/ceph/chpc/shared/janine_bijsterbosch_group/lucas_scratch/SiT_structural_encoder/patch_extraction"
data_destination_path = "/ceph/chpc/shared/janine_bijsterbosch_group/lucas_scratch/HCP_YA"

scans_ids = np.array([f for f in os.listdir(subject_path) if f.isnumeric()])
print(len(scans_ids))
indices_mesh_triangles=pd.read_csv(f'{patch_extraction_path}/triangle_indices_ico_{ico_res}_sub_ico_{ico_down_res}.csv')


1113


In [4]:
def preprocess(ids):
    num_subjects = len(ids)

    data = []
    for i, id in enumerate(ids):
        L = f"{subject_path}/{id}/{id}_ico{ico_res}.L.shape.gii"
        data.append(np.array(nib.load(L).agg_data())) # in the future, w/ more channels we can change this
    
        R = f"{subject_path}/{id}/{id}_ico{ico_res}.R.shape.gii"
        data.append(np.array(nib.load(R).agg_data()))

    ## NORMALIZE THE DATA

    data = np.asarray(data)
    normalised_data = data # if I find any way to normalize it we can put it here


    data = np.zeros((num_subjects*2, 1, down_num_patches, down_num_vertices)) # 1 == number of channels
    for i, id in enumerate(scans_ids):
        for j in range(down_num_patches):
            indices_to_extract = indices_mesh_triangles[str(j)].to_numpy()
            data[i,:,j,:] = normalised_data[2*i][indices_to_extract]
            data[i+num_subjects,:,j,:] = normalised_data[2*i+1][indices_to_extract]

            # what we see here: store left side at index i, right side at index i + num_subjects
    return data

In [4]:
data = preprocess(scans_ids)

In [7]:
filename = os.path.join(data_destination_path, f"ico{ico_res}_to_{ico_down_res}_patch_data.npy")
np.save(filename, data)